# Pre-training & Detailed Evaluation on INCLUDE Dataset
### Stage 1 Visual Backbone (ResNet-18 + GCMM + GAM)

This notebook trains and performs detailed evaluation of GANet's Stage 1 visual modeling on the **INCLUDE** isolated Indian Sign Language dataset.

**Outputs Produced:**
1. **`work_dir/include_pretrain/include_visual.pt`**: Checkpoint containing trained Stage 1 visual weights ready for ISL-CSLTR transfer.
2. **`work_dir/include_pretrain/include_results.json`**: Top-1 and Top-5 accuracy logs per epoch.
3. **`work_dir/include_pretrain/include_per_class_metrics.csv`**: Detailed per-sign accuracy across all classes.

### Step 1: Check GPU Acceleration

In [ ]:
!nvidia-smi

### Step 2: Unpack Codebase
Upload `ganet_code.zip` via Kaggle **[+ Add Input]**.

In [ ]:
import os, glob, shutil

!mkdir -p /kaggle/working/ganet_isl

code_zips = glob.glob('/kaggle/input/**/ganet_code*.zip', recursive=True) + glob.glob('/kaggle/working/**/ganet_code*.zip', recursive=True)
code_dirs = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/train_isl.py', recursive=True)]

if code_zips:
    print(f"[*] Unpacking codebase from: {code_zips[0]}")
    !unzip -qo {code_zips[0]} -d /kaggle/working/ganet_isl/
elif code_dirs:
    print(f"[*] Copying unzipped codebase from: {code_dirs[0]}")
    !cp -rf {code_dirs[0]}/* /kaggle/working/ganet_isl/
else:
    print("[!] Codebase not found! Upload ganet_code.zip.")

%cd /kaggle/working/ganet_isl
!pip install -q opencv-python pyyaml tqdm pandas
print("[✓] Workspace ready at:", os.getcwd())

### Step 3: Run Detailed Training & Evaluation on INCLUDE

In [ ]:
%cd /kaggle/working/ganet_isl

# Trains Stage 1 with Top-1 / Top-5 metrics and exports per-class CSV
!python train_include_detailed.py --config configs/include.yaml

### Step 4: Display Summary & Top/Bottom Performing Signs

In [ ]:
%cd /kaggle/working/ganet_isl
import json, os
import pandas as pd

res_path = 'work_dir/include_pretrain/include_results.json'
csv_path = 'work_dir/include_pretrain/include_per_class_metrics.csv'

if os.path.exists(res_path):
    res = json.load(open(res_path))
    print("=" * 50)
    print("INCLUDE FINAL EVALUATION SUMMARY")
    print("=" * 50)
    print(f"Best Validation Top-1 Acc: {res.get('best_val_top1', 0.0):.2f}%")
    print(f"Test Top-1 Accuracy:       {res.get('test_top1', 0.0):.2f}%")
    print(f"Test Top-5 Accuracy:       {res.get('test_top5', 0.0):.2f}%")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("\n--- 10 Most Accurately Recognized Signs ---")
    print(df.sort_values(by='accuracy_pct', ascending=False).head(10).to_string(index=False))
    
    print("\n--- 10 Most Challenging Signs ---")
    print(df.sort_values(by='accuracy_pct', ascending=True).head(10).to_string(index=False))